# OTP vs GTFS comparison notebook

This notebook helps you inspect what OTP ingested vs your GTFS CSVs.

How to use:
- Make sure your OTP server is running on http://localhost:8080 (as you already have).
- Run cells from top to bottom.
- Configure `endpoint`, `gtfs_zip_path`, `route_gtfs_id`, and `trip_gtfs_id` below.

It will:
- Fetch stops and trip pattern stops from OTP (GTFS GraphQL schema).
- Unzip and load your GTFS `stops.txt`, `trips.txt`, and `stop_times.txt`.
- Compare stop names/coordinates and a trip’s stop sequence.



In [13]:
# Single-run pipeline cell (top-to-bottom safe)
from pathlib import Path
import zipfile
import requests
import pandas as pd

# Config (safe defaults; override above if already defined)
try:
    endpoint
except NameError:
    endpoint = "http://localhost:8080/otp/gtfs/v1"
try:
    project_root
except NameError:
    project_root = Path.cwd().parent
try:
    gtfs_zip_path
except NameError:
    gtfs_zip_path = project_root / "otp-exploration" / "graphs" / "luxembourg" / "gtfs.zip"
try:
    route_gtfs_id
except NameError:
    route_gtfs_id = "1:2679"
try:
    trip_gtfs_id
except NameError:
    trip_gtfs_id = "1:20857102"

# Ensure endpoint is reachable with a trivial query
probe = {"query": "{ routes { id } }"}
r = requests.post(endpoint, json=probe, timeout=20)
r.raise_for_status()
_ = r.json()
print("Endpoint OK:", endpoint)

# GraphQL helper and queries (String! for trip id)

def gql(query: str, variables: dict | None = None) -> dict:
    resp = requests.post(endpoint, json={"query": query, "variables": variables or {}}, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    if "errors" in data:
        raise RuntimeError(f"GraphQL errors: {data['errors']}")
    return data["data"]

Q_STOPS = """
{ stops { id name lat lon gtfsId } }
"""

Q_TRIP_PATTERN_STOPS = """
query($tripId: String!) {
  trip(id: $tripId) {
    id
    gtfsId
    pattern { id name stops { id gtfsId name lat lon } }
  }
}
"""

# Unzip GTFS
extract_dir = project_root / "notebooks" / ".cache" / "gtfs"
extract_dir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(gtfs_zip_path, 'r') as zf:
    zf.extractall(extract_dir)
print("GTFS extracted to:", extract_dir)

# Load CSVs - use dtype to preserve leading zeros in stop_id and trip_id
stops_txt = pd.read_csv(extract_dir / "stops.txt", dtype={'stop_id': str})
trips_txt = pd.read_csv(extract_dir / "trips.txt", dtype={'trip_id': str})
stop_times_txt = pd.read_csv(extract_dir / "stop_times.txt", dtype={'trip_id': str, 'stop_id': str})

# Fetch OTP data
stops_data = gql(Q_STOPS)["stops"]
print("Fetched stops:", len(stops_data))
trip_data = gql(Q_TRIP_PATTERN_STOPS, {"tripId": trip_gtfs_id})["trip"]
print("Trip pattern:", trip_data["pattern"]["name"], "stops:", len(trip_data["pattern"]["stops"]))

# Build OTP stops dataframe and cleaned id columns
otp_stops = pd.DataFrame(stops_data)
# Keep original gtfsId but create a cleaned stop id (remove feed prefix)
otp_stops['stop_id_clean'] = otp_stops['gtfsId'].astype(str).str.split(":", n=1).str[-1].str.strip()

# No need to convert stop_id to str again since we loaded it as str
stops_txt['stop_id_clean'] = stops_txt['stop_id'].str.split(":", n=1).str[-1].str.strip()

# Diagnostics before merge
print("\nDiagnostics:")
print("Sample OTP stop_id_clean:", otp_stops['stop_id_clean'].head().tolist())
print("Sample GTFS stop_id_clean:", stops_txt['stop_id_clean'].head().tolist())

# Merge using cleaned keys to avoid prefix mismatches
merged_stops = otp_stops.merge(
    stops_txt[["stop_id_clean", "stop_name", "stop_lat", "stop_lon"]],
    left_on="stop_id_clean", right_on="stop_id_clean", how="left"
)
# Compute deltas safely (coerce missing values to NaN)
merged_stops['delta_lat'] = pd.to_numeric(merged_stops['lat'], errors='coerce') - pd.to_numeric(merged_stops['stop_lat'], errors='coerce')
merged_stops['delta_lon'] = pd.to_numeric(merged_stops['lon'], errors='coerce') - pd.to_numeric(merged_stops['stop_lon'], errors='coerce')
print("\nStops merged (clean keys):", merged_stops.shape)
print("Missing stop_name after clean merge:", merged_stops['stop_name'].isna().sum())
display(merged_stops.head(10))

# Compare trip stop sequence using cleaned ids
pattern_stops = pd.DataFrame(trip_data['pattern']['stops']).copy()
pattern_stops['stop_id_clean'] = pattern_stops['gtfsId'].astype(str).str.split(":", n=1).str[-1].str.strip()

# No need to convert stop_id/trip_id to str again since we loaded them as str
stop_times_txt['stop_id_clean'] = stop_times_txt['stop_id'].str.split(":", n=1).str[-1].str.strip()
stop_times_txt['trip_id_clean'] = stop_times_txt['trip_id'].str.split(":", n=1).str[-1].str.strip()

# Filter stop_times by numeric trip id (CSV side uses numeric id)
trip_id_clean = str(trip_gtfs_id).split(":", 1)[-1]
seq_gtfs = (
    stop_times_txt.loc[stop_times_txt['trip_id_clean'] == trip_id_clean,
                       ['stop_id_clean', 'stop_sequence', 'arrival_time', 'departure_time']]
    .sort_values('stop_sequence')
    .reset_index(drop=True)
)

# More diagnostics
print("\nTrip stop sequence diagnostics:")
print("Sample pattern stop_id_clean:", pattern_stops['stop_id_clean'].head().tolist())
print("Sample seq_gtfs stop_id_clean:", seq_gtfs['stop_id_clean'].head().tolist())

# Merge pattern stops with seq_gtfs on the cleaned stop id
joined = pattern_stops.merge(seq_gtfs, left_on='stop_id_clean', right_on='stop_id_clean', how='left')
print("\nPattern rows:", len(pattern_stops))
print("Rows matched in GTFS stop_times:", joined['stop_sequence'].notna().sum())
print("Pattern rows missing stop_times match:", joined['stop_sequence'].isna().sum())
display(joined.head(20))

# Show unmatched pattern stops for inspection
if joined['stop_sequence'].isna().any():
    print("\nUnmatched pattern stops:")
    display(joined.loc[joined['stop_sequence'].isna(), ['gtfsId', 'stop_id_clean']].head(20))

Endpoint OK: http://localhost:8080/otp/gtfs/v1
GTFS extracted to: c:\Users\Usuario\Desktop\UPF\4t\TFG\route-planner-tfg\notebooks\.cache\gtfs
GTFS extracted to: c:\Users\Usuario\Desktop\UPF\4t\TFG\route-planner-tfg\notebooks\.cache\gtfs
Fetched stops: 2805
Trip pattern: 653 stops: 20

Diagnostics:
Sample OTP stop_id_clean: ['000110101001', '000110101002', '000110101004', '000110101008', '000110101009']
Sample GTFS stop_id_clean: ['000200403005', '000200402003', '000200402004', '000200304002', '000200417037']
Fetched stops: 2805
Trip pattern: 653 stops: 20

Diagnostics:
Sample OTP stop_id_clean: ['000110101001', '000110101002', '000110101004', '000110101008', '000110101009']
Sample GTFS stop_id_clean: ['000200403005', '000200402003', '000200402004', '000200304002', '000200417037']

Stops merged (clean keys): (2805, 11)
Missing stop_name after clean merge: 0

Stops merged (clean keys): (2805, 11)
Missing stop_name after clean merge: 0


,id,name,lat,lon,gtfsId,stop_id_clean,stop_name,stop_lat,stop_lon,delta_lat,delta_lon
0,U3RvcDoxOjAwMDExMDEwMTAwMQ,"Clervaux, Clinique",50.053323,6.025082,1:000110101001,000110101001,"Clervaux, Clinique",50.053323,6.025082,0.0,0.0
1,U3RvcDoxOjAwMDExMDEwMTAwMg,"Clervaux, Gare routière/Lycée",50.063617,6.024306,1:000110101002,000110101002,"Clervaux, Gare routière/Lycée",50.063617,6.024306,0.0,0.0
2,U3RvcDoxOjAwMDExMDEwMTAwNA,"Clervaux, Maison de Retraite",50.057799,6.025616,1:000110101004,000110101004,"Clervaux, Maison de Retraite",50.057799,6.025616,0.0,0.0
3,U3RvcDoxOjAwMDExMDEwMTAwOA,"Clervaux, Place de la Libération",50.052878,6.029981,1:000110101008,000110101008,"Clervaux, Place de la Libération",50.052878,6.029981,0.0,0.0
4,U3RvcDoxOjAwMDExMDEwMTAwOQ,"Clervaux, Place Benelux",50.054794,6.031070,1:000110101009,000110101009,"Clervaux, Place Benelux",50.054794,6.031070,0.0,0.0
5,U3RvcDoxOjAwMDExMDEwMTAxMA,"Clervaux, Police",50.054141,6.025698,1:000110101010,000110101010,"Clervaux, Police",50.054141,6.025698,0.0,0.0
6,U3RvcDoxOjAwMDExMDEwMTAxMQ,"Clervaux, Postes",50.054836,6.030853,1:000110101011,000110101011,"Clervaux, Postes",50.054836,6.030853,0.0,0.0
7,U3RvcDoxOjAwMDExMDEwMTAxMw,"Clervaux, Baastnecherstrooss",50.064428,6.022033,1:000110101013,000110101013,"Clervaux, Baastnecherstrooss",50.064428,6.022033,0.0,0.0
8,U3RvcDoxOjAwMDExMDEwMTAxNA,"Clervaux, Kalborn/Kaalber",50.101920,6.111628,1:000110101014,000110101014,"Clervaux, Kalborn/Kaalber",50.101920,6.111628,0.0,0.0
9,U3RvcDoxOjAwMDExMDEwMTAxNQ,"Clervaux, Hall Polyvalent",50.061290,6.018076,1:000110101015,000110101015,"Clervaux, Hall Polyvalent",50.061290,6.018076,0.0,0.0



Trip stop sequence diagnostics:
Sample pattern stop_id_clean: ['000220903014', '000220903007', '000220903025', '000220903003', '000220903022']
Sample seq_gtfs stop_id_clean: ['000220903014', '000220903007', '000220903025', '000220903003', '000220903022']

Pattern rows: 20
Rows matched in GTFS stop_times: 20
Pattern rows missing stop_times match: 0


,id,gtfsId,name,lat,lon,stop_id_clean,stop_sequence,arrival_time,departure_time
0,U3RvcDoxOjAwMDIyMDkwMzAxNA,1:000220903014,"Rodange, Am Duerf",49.545474,5.839345,000220903014,0,7:25:00,7:25:00
1,U3RvcDoxOjAwMDIyMDkwMzAwNw,1:000220903007,"Rodange, Klopp",49.544964,5.837802,000220903007,1,7:25:30,7:25:30
2,U3RvcDoxOjAwMDIyMDkwMzAyNQ,1:000220903025,"Rodange, Waeschbuer",49.544474,5.832386,000220903025,2,7:26:00,7:26:00
3,U3RvcDoxOjAwMDIyMDkwMzAwMw,1:000220903003,"Rodange, Croix St. Pierre",49.543377,5.828489,000220903003,3,7:26:30,7:26:30
4,U3RvcDoxOjAwMDIyMDkwMzAyMg,1:000220903022,"Rodange, Siole",49.542234,5.824844,000220903022,4,7:27:00,7:27:00
5,U3RvcDoxOjAwMDIyMDkwMzAwNQ,1:000220903005,"Rodange, Fontaine d'Olière",49.542422,5.822536,000220903005,5,7:27:30,7:27:30
6,U3RvcDoxOjAwMDIyMDkwMzAwNA,1:000220903004,"Rodange, Fonderie",49.545658,5.824386,000220903004,6,7:28:00,7:28:00
7,U3RvcDoxOjAwMDIyMDkwMzAwMg,1:000220903002,"Rodange, Cité à la Wissi",49.547032,5.827906,000220903002,7,7:29:00,7:29:00
8,U3RvcDoxOjAwMDIyMDkwMzAyMA,1:000220903020,"Rodange, Schlaakemillen Wissi",49.548636,5.831937,000220903020,8,7:30:00,7:30:00
9,U3RvcDoxOjAwMDIyMDkwMzAwOQ,1:000220903009,"Rodange, Gendarmerie",49.549944,5.838296,000220903009,9,7:31:00,7:31:00


In [7]:
stops_txt

,stop_id,stop_code,stop_name,stop_desc,stop_lat,stop_lon,location_type,parent_station,wheelchair_boarding,platform_code,stop_id_clean
0,200403005,NaN,"Belair, Sacré-Coeur",NaN,49.610276,6.113159,0,NaN,0,NaN,200403005
1,200402003,NaN,"Beggen, Cyprien Merjai",NaN,49.643399,6.134779,0,NaN,0,NaN,200402003
2,200402004,NaN,"Beggen, Henri Dunant",NaN,49.647594,6.134546,0,NaN,0,NaN,200402004
3,200304002,NaN,"Howald, Bei der Kierch",NaN,49.583246,6.142332,0,NaN,0,NaN,200304002
4,200417037,NaN,"Kirchberg, Schmëdd",NaN,49.628363,6.149234,0,NaN,0,NaN,200417037
...,...,...,...,...,...,...,...,...,...,...,...
2800,150607005,NaN,"Reichlange, Bréck",NaN,49.774840,5.925410,0,NaN,0,NaN,150607005
2801,220402068,NaN,"Esch-sur-Alzette, poteau de Kayl",NaN,49.485275,6.004235,0,NaN,0,NaN,220402068
2802,210502003,NaN,"Ellange, rue de la Gare",NaN,49.516347,6.303918,0,NaN,0,NaN,210502003
2803,220804015,NaN,"Pontpierre, Europastrooss",NaN,49.540725,6.033593,0,NaN,0,NaN,220804015


In [12]:
otp_stops[otp_stops["name"]=="Belair, Sacré-Coeur"]

,id,name,lat,lon,gtfsId,stop_id_clean
1425,U3RvcDoxOjAwMDIwMDQwMzAwNQ,"Belair, Sacré-Coeur",49.610276,6.113159,1:000200403005,000200403005
